In [3]:
import gym
from gym import spaces
import numpy as np
from ObservationAdaptors import *
from VehicleControl import *
from LoadOpenDrive2 import *
from ObjectSpawn import *
from LoadOpenDrive2 import *
import time

import carla, random

SUPPORTED_SIGNS_COUNT = 5

class CarlaEnv(gym.Env):
    def __init__(self, map_path, walkers_count, vehicles_count, max_steps=50000000):
        super(CarlaEnv, self).__init__()
        
        self.walkers_count = walkers_count
        self.vehicles_count = vehicles_count
        
        load_opendrive_map(map_path)
        
        self.client = carla.Client('localhost', 2000)
        self.world = self.client.get_world()
        self.ego_vehicle = spawn_ego_vehicle(self.world)
        self.vehicle_controller = VehicleController(self.world, self.ego_vehicle)
        spawn_vehicles(self.client, vehicles_count)
        self.walkers = spawn_pedestrians(self.world, walkers_count)
        self.max_steps = max_steps
        self.current_step = 0

        self.action_space = spaces.MultiDiscrete([3, 3])

        self.observation_space = spaces.Dict({
            "speed_x": spaces.Box(low=-np.inf, high=np.inf, shape=(3, 6), dtype=np.float32),
            "speed_y": spaces.Box(low=-np.inf, high=np.inf, shape=(3, 6), dtype=np.float32),
            "presence": spaces.Box(low=0, high=1, shape=(3, 6), dtype=np.float32),
            "lane_angle": spaces.Box(low=-np.pi, high=np.pi, shape=(), dtype=np.float32),
            "max_speed": spaces.Box(low=0, high=200, shape=(), dtype=np.float32),
            "traffic_signs": spaces.Box(low=0, high=1, shape=(SUPPORTED_SIGNS_COUNT,), dtype=np.float32),  # SUPPORTED_SIGNS_COUNT traffic signs encoded as one-hot
        })
        self.reset()

    def reset(self):
        self.current_step = 0
        destroy_all_actors(self.client)
        self.ego_vehicle = spawn_ego_vehicle(self.world)
        self.vehicle_controller = VehicleController(self.world, self.ego_vehicle)
        spawn_vehicles(self.client, self.vehicles_count)
        self.walkers = spawn_pedestrians(self.world, self.walkers_count)
        self.current_step = 0
        return self._get_observation()

    def step(self, speed_action, turn_action):
        self.vehicle_controller.exec_command(self.vehicle_controller.speed_action_convertor(speed_action))
        self.vehicle_controller.exec_command(self.vehicle_controller.turn_action_convertor(turn_action))

        prev_obs = self._get_observation()

        self.world.tick()
        # Step pedestrians
        step_peds(self.world, self.walkers)

        # Calculate reward
        reward = self.vehicle_controller.get_reward()

        # Additional penalty or reward for traffic signs
        traffic_signs = self._get_nearby_traffic_signs()
        reward += self._process_traffic_signs(traffic_signs)

        # Get observation
        obs = self._get_observation()

        # Check if done
        self.current_step += 1
        done = self.current_step >= self.max_steps

        # Return step info
        return obs, reward, done, {}

    def _get_observation(self):
        x_speed_matrix, y_speed_matrix, presence_matrix = get_speed_matrices(self.ego_vehicle)
        lane_angle = get_lane_angle(self.ego_vehicle, self.world.get_map())
        traffic_signs = self._encode_traffic_signs()

        return {
            "speed_x": x_speed_matrix,
            "speed_y": y_speed_matrix,
            "presence": presence_matrix,
            "lane_angle": lane_angle,
            "traffic_signs": traffic_signs,
        }

    def _get_nearby_traffic_signs(self):
        return get_nearby_signs(self.ego_vehicle, self.world.get_map(), radius=10)

    def _encode_traffic_signs(self):
        traffic_signs = self._get_nearby_traffic_signs()
        encoded_signs = np.zeros(10)  # Assuming 10 possible traffic sign types

        for sign in traffic_signs:
            if sign.type.isdigit():
                sign_index = int(sign.type) % 10  # Map sign type to an index
                encoded_signs[sign_index] = 1

        return encoded_signs

    def _process_traffic_signs(self, traffic_signs):
        # penalty = 0
        # for sign in traffic_signs:
        #     if sign.type == "1000001":  # Example: stop sign
        #         penalty -= 5 if self.vehicle_controller.control.throttle > 0 else 0
        #     elif sign.type == "1000002":  # Example: speed limit
        #         speed = self.ego_vehicle.get_velocity()
        #         speed_kmh = 3.6 * ((speed.x**2 + speed.y**2)**0.5)
        #         if speed_kmh > 50:  # Assuming speed limit of 50 km/h
        #             penalty -= 1
        # return penalty
        return 0

    def render(self, mode="human"):
        pass  # Visualization logic can go here if needed

    def close(self):
        pass
from stable_baselines3 import SAC
from stable_baselines3.common.env_checker import check_env

def run(map_path, walkers_count, vehicles_count, steps, device):
    env = CarlaEnv(map_path, walkers_count, vehicles_count, max_steps=steps)
    check_env(env, warn=True)  
    model = SAC("MlpPolicy", env, verbose=1, tensorboard_log="./sac_carla/", device=device)
    model.learn(total_timesteps=100000)
    model.save("sac_carla_model")
map_path = "C:/Users/H/Desktop/IOT/Carla-Integration-Modules/LoadOpenDrive2/simple_map.xodr"
run(map_path, 10, 10, 10000, "cuda")

In [4]:
from stable_baselines3 import SAC
from stable_baselines3.common.env_checker import check_env

def run(map_path, walkers_count, vehicles_count, steps, device):
    env = CarlaEnv(map_path, walkers_count, vehicles_count, max_steps=steps)
    check_env(env, warn=True)  
    model = SAC("MlpPolicy", env, verbose=1, tensorboard_log="./sac_carla/", device=device)
    model.learn(total_timesteps=100000)
    model.save("sac_carla_model")

In [ ]:
map_path = "C:/Users/H/Desktop/IOT/Carla-Integration-Modules/LoadOpenDrive2/simple_map.xodr"
run(map_path, 10, 10, 10000, "cuda")

Map successfully loaded into CARLA.
Spawned vehicle 34 at Location(x=-83.027412, y=-4.544271, z=0.500000)
Spawned vehicle 35 at Location(x=1.495843, y=2.727994, z=0.500000)
Spawned vehicle 36 at Location(x=-12.508623, y=18.080153, z=0.500000)
Spawned vehicle 37 at Location(x=-69.429817, y=34.672935, z=0.500000)
Spawned vehicle 38 at Location(x=-86.217575, y=-4.794966, z=0.500000)
Spawned vehicle 39 at Location(x=-62.864815, y=38.906181, z=0.500000)
Spawned vehicle 40 at Location(x=-13.108853, y=-48.354305, z=0.500000)
Spawned vehicle 41 at Location(x=-5.593005, y=-48.094692, z=0.500000)
Spawned pedestrain 42 at Location(x=-63.046944, y=42.100994, z=0.500000)
Spawned pedestrain 43 at Location(x=-13.204116, y=12.827237, z=0.500000)
Spawned pedestrain 44 at Location(x=-1.434308, y=1.441832, z=0.500000)
Spawned pedestrain 45 at Location(x=-1.434308, y=1.441832, z=0.500000)
Spawned pedestrain 46 at Location(x=13.562593, y=-31.204296, z=0.500000)
Spawned pedestrain 47 at Location(x=-65.72061

AssertionError: Your environment must inherit from the gymnasium.Env class cf. https://gymnasium.farama.org/api/env/

: 

In [17]:
import gymnasium as gym
from gymnasium.spaces import MultiDiscrete

action_space = MultiDiscrete([3,3])
for i in range(15):
    print (f'{i}th sample action : {action_space.sample()}')

0th sample action : [1 0]
1th sample action : [0 1]
2th sample action : [2 0]
3th sample action : [2 0]
4th sample action : [1 0]
5th sample action : [2 0]
6th sample action : [1 0]
7th sample action : [0 2]
8th sample action : [0 0]
9th sample action : [1 0]
10th sample action : [0 0]
11th sample action : [1 2]
12th sample action : [1 0]
13th sample action : [0 2]
14th sample action : [2 1]
